In [ ]:
train_df = pd.read_excel('/kaggle/input/datasets/mohamed00mamdouh/deepxdata/DeepX_train.xlsx')
val_df = pd.read_excel('/kaggle/input/datasets/mohamed00mamdouh/deepxdata/DeepX_validation.xlsx')

In [ ]:
import pandas as pd
import ast
import json
import re
import unicodedata

In [ ]:
aspect_cols = [
    "food",
    "service",
    "price",
    "cleanliness",
    "delivery",
    "ambiance",
    "app_experience",
    "general",
    "none"
]

In [ ]:
def clean_multilingual_text(text):
    if text is None:
        return ""

    text = str(text)

    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)

    text = re.sub(r"[إأآٱ]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ؤ", "و", text)
    text = re.sub(r"ئ", "ي", text)

    arabic_diacritics = re.compile(r"[\u0617-\u061A\u064B-\u0652]")
    text = re.sub(arabic_diacritics, "", text)

    text = re.sub(r"(.)\1{2,}", r"\1\1", text)
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
def safe_parse(x):
    if isinstance(x, (list, dict)):
        return x
    if pd.isna(x):
        return None
    if isinstance(x, str):
        x = x.strip()
        try:
            return ast.literal_eval(x)
        except:
            try:
                return json.loads(x)
            except:
                return None
    return None

In [ ]:
def prepare_absa_data(origin_df):
    df = origin_df.copy()

    # Parse
    df["aspects"] = df["aspects"].apply(safe_parse)
    df["aspect_sentiments"] = df["aspect_sentiments"].apply(safe_parse)

    # Clean text
    df["clean_text"] = df["review_text"].apply(clean_multilingual_text)

    # Aspect columns

    for col in aspect_cols:
        df[col] = 0

    for i, row in df.iterrows():
        aspects_list = row["aspects"]

        if not isinstance(aspects_list, list) or len(aspects_list) == 0:
            df.at[i, "none"] = 1
        else:
            for a in aspects_list:
                if a in aspect_cols:
                    df.at[i, a] = 1


    # Aspect DF
    aspect_df = df[["review_id", "review_text", "clean_text"] + aspect_cols].copy()

    aspect_df = aspect_df.rename(columns={
        "review_id": "id",
        "review_text": "original_text",
        "clean_text": "text_for_aspect"
    })


    # Sentiment DF

    rows = []

    for _, row in df.iterrows():
        review_id = row["review_id"]
        original_text = row["review_text"]
        clean_text = row["clean_text"]
        aspects = row["aspects"]
        aspect_sentiments = row["aspect_sentiments"]

        if not isinstance(aspects, list):
            continue
        if not isinstance(aspect_sentiments, dict):
            continue

        for aspect in aspects:
            if aspect in aspect_sentiments:
                rows.append({
                    "id": review_id,
                    "original_text": original_text,
                    "text_for_sentiment": f"{clean_text} [ASPECT] {aspect}",
                    "aspect": aspect,
                    "label_for_sentiment": aspect_sentiments[aspect]
                })

    sentiment_df = pd.DataFrame(rows)

    return aspect_df, sentiment_df

In [ ]:
aspect_train_df, sentiment_train_df = prepare_absa_data(train_df)
aspect_val_df, sentiment_val_df = prepare_absa_data(val_df)

In [ ]:
aspect_train_df.to_csv('aspect_train_df.csv', index=False)
aspect_val_df.to_csv('aspect_val_df.csv', index=False)

sentiment_train_df.to_csv('sentiment_train_df.csv', index=False)
sentiment_val_df.to_csv('sentiment_val_df.csv', index=False)

In [ ]:
aspect_train_df.head()

In [ ]:
sentiment_train_df.head()